### Importing Libraries and loading data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

train_df = pd.read_csv('../datasets/cleaned datasets/train.csv')
test_df = pd.read_csv('../datasets/cleaned datasets/test.csv')

In [4]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 12 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           750000 non-null  int64  
 1   Podcast_Name                 750000 non-null  object 
 2   episode_number               750000 non-null  int64  
 3   Episode_Length_minutes       750000 non-null  float64
 4   Genre                        750000 non-null  object 
 5   Host_Popularity_percentage   750000 non-null  float64
 6   Publication_Day              750000 non-null  object 
 7   Publication_Time             750000 non-null  object 
 8   Guest_Popularity_percentage  750000 non-null  float64
 9   Number_of_Ads                750000 non-null  float64
 10  Episode_Sentiment            750000 non-null  object 
 11  Listening_Time_minutes       750000 non-null  float64
dtypes: float64(5), int64(2), object(5)
memory usage: 68.7+ MB


In [5]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 11 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           250000 non-null  int64  
 1   Podcast_Name                 250000 non-null  object 
 2   episode_number               250000 non-null  int64  
 3   Episode_Length_minutes       250000 non-null  float64
 4   Genre                        250000 non-null  object 
 5   Host_Popularity_percentage   250000 non-null  float64
 6   Publication_Day              250000 non-null  object 
 7   Publication_Time             250000 non-null  object 
 8   Guest_Popularity_percentage  250000 non-null  float64
 9   Number_of_Ads                250000 non-null  float64
 10  Episode_Sentiment            250000 non-null  object 
dtypes: float64(4), int64(2), object(5)
memory usage: 21.0+ MB


### Publication day and publication time concat

We can make a combo column, combining publication day and publication time together.
For example, if its published on weekend + evening or night, i'd assume it would have a greater impact on the average listenings. 

In [15]:
def fe_publication_day_time_combo(row):
    if row['Publication_Day'] in ['Saturday', 'Sunday', 'Friday'] and row['Publication_Time'] in ['Evening', 'Night']:
        return 1
    else:
        return 0

train_df['peak_time_to_publish'] = train_df.apply(fe_publication_day_time_combo, axis=1)
test_df['peak_time_to_publish'] = test_df.apply(fe_publication_day_time_combo, axis=1)

### we will also perform the same for publication time and day alone

In [18]:
def fe_is_nigth_or_evening(row):
    if row['Publication_Time'] in ['Evening', 'Night']:
        return 1
    else:
        return 0

train_df['is_night_or_evening'] = train_df.apply(fe_is_nigth_or_evening, axis=1)
test_df['is_night_or_evening'] = test_df.apply(fe_is_nigth_or_evening, axis=1)

In [19]:
def fe_is_weekend(row):
    if row['Publication_Day'] in ['Friday','Saturday', 'Sunday']:
        return 1
    else:
        return 0

train_df['is_weekend'] = train_df.apply(fe_is_weekend, axis=1)
test_df['is_weekend'] = test_df.apply(fe_is_weekend, axis=1)

### We will create a feature called 'peak_popularity' and this will get the value 1 if both the guest and the host has a popularity above 80%

In [22]:
def fe_peak_poluarity(row):
    if row['Host_Popularity_percentage'] >= 80 and row['Guest_Popularity_percentage'] >= 80:
        return 1
    else:
        return 0

train_df['peak_popularity'] = train_df.apply(fe_peak_poluarity, axis=1)
test_df['peak_popularity'] = test_df.apply(fe_peak_poluarity, axis=1)

### Creating an 'addless' feature

In [25]:
def fe_addless(row):
    if row['Number_of_Ads'] == 0:
        return 1
    else:
        return 0

train_df['is_addless'] = train_df.apply(fe_addless, axis=1)
test_df['is_addless'] = test_df.apply(fe_addless, axis=1)

### is_midweek, is_monday

In [38]:
def fe_is_midweek(row):
    if row['Publication_Day'] in ['Tuesday', 'Wednesday', 'Thursday']:
        return 1
    else:
        return 0

train_df['is_midweek'] = train_df.apply(fe_is_midweek, axis=1)
test_df['is_midweek'] = test_df.apply(fe_is_midweek, axis=1)

In [39]:
def fe_is_monday(row):
    if row['Publication_Day'] == 'Monday':
        return 1
    else:
        return 0

train_df['is_monday'] = train_df.apply(fe_is_monday, axis=1)
test_df['is_monday'] = test_df.apply(fe_is_monday, axis=1)

### Relative popularity

In [40]:
def fe_total_popularity(row):
    host_p = row['Host_Popularity_percentage']
    guest_p = row['Guest_Popularity_percentage']
    total_p = host_p + guest_p
    return total_p


def fe_diff_popularity(row):
    host_p = row['Host_Popularity_percentage']
    guest_p = row['Guest_Popularity_percentage']
    difference_p = host_p - guest_p
    return difference_p


train_df['total_popularity'] = train_df.apply(fe_total_popularity, axis=1)
test_df['total_popularity'] = test_df.apply(fe_total_popularity, axis=1)


train_df['popularity_difference'] = train_df.apply(fe_diff_popularity, axis=1)
test_df['popularity_difference'] = test_df.apply(fe_diff_popularity, axis=1)

### Ad density

In [41]:
def fe_ad_density(row):
    ad_count = row['Number_of_Ads']
    length = row['Episode_Length_minutes']
    if ad_count == 0:
        return 0
    else:
        density = ad_count / length
        return density

train_df['ad_density'] = train_df.apply(fe_ad_density, axis=1)
test_df['ad_density'] = train_df.apply(fe_ad_density, axis=1)

In [42]:
train_df.to_csv('../datasets/cleaned datasets/train.csv', index=None)
test_df.to_csv('../datasets/cleaned datasets/test.csv', index=None)